In [5]:
import os
import numpy
import pandas as pd
import numpy as np
from scipy.spatial.distance import cosine
from scipy.stats import spearmanr, pearsonr
import nltk

embeddings_index = {}
with open('glove/glove.6B.100d.txt', encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

print(f'Found {len(embeddings_index)} word vectors.')

wordsim = pd.read_csv('benchmark/wordsim-353/wordsim353.csv')
wordsim.columns = ['word1', 'word2', 'human_score']

counter_embeddings = {}
with open('counter-fitting/results/counter_fitted_vectors.txt', encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        counter_embeddings[word] = coefs

print(f'Found {len(counter_embeddings)} word vectors.')

retro_embeddings = {}
with open('r_glove20.txt', encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        retro_embeddings[word] = coefs

print(f'Found {len(retro_embeddings)} word vectors.')

Found 400000 word vectors.
Found 400000 word vectors.
Found 400000 word vectors.


In [6]:
import numpy as np
from scipy.spatial.distance import cosine
from scipy.stats import spearmanr, pearsonr

def wordsim_analysis(embeddings_dict, wordsim_df=wordsim, lowercase=True):
    from scipy.spatial.distance import cosine
    from scipy.stats import spearmanr, pearsonr

    def cosine_similarity(vec1, vec2):
        return 1 - cosine(vec1, vec2)
    
    predicted_scores = []
    human_scores = []
    
    for _, row in wordsim_df.iterrows():
        w1 = row['word1']
        w2 = row['word2']
        human_score = row['human_score']
        
        if lowercase:
            w1, w2 = w1.lower(), w2.lower()
        
        if w1 in embeddings_dict and w2 in embeddings_dict:
            sim = cosine_similarity(embeddings_dict[w1], embeddings_dict[w2])
            predicted_scores.append(sim)
            human_scores.append(human_score)
    
    spearman_corr, _ = spearmanr(human_scores, predicted_scores)
    pearson_corr, _ = pearsonr(human_scores, predicted_scores)
    
    metrics = {
        'pairs_evaluated': len(predicted_scores),
        'total_pairs': len(wordsim_df),
        'spearman_corr': spearman_corr,
        'pearson_corr': pearson_corr
    }
    
    return metrics, human_scores, predicted_scores

In [7]:
def simlex_analysis(word_vectors):
    """
    Compute Spearman's rho between the gold SimLex-999 scores and
    the cosine similarity of the supplied word vectors.
    """
    simlex_path = "counter-fitting/linguistic_constraints/SimLex-999.txt"
    pair_list = []

    with open(simlex_path, "r", encoding="utf-8") as fread_simlex:
        next(fread_simlex)  # skip header
        for line in fread_simlex:
            tokens = line.strip().split()
            if len(tokens) < 4:
                continue
            word_i = tokens[0].lower()
            word_j = tokens[1].lower()
            score = float(tokens[3])  # human similarity score
            if word_i in word_vectors and word_j in word_vectors:
                pair_list.append((word_i, word_j, score))

    if len(pair_list) < 2:
        print("Too few SimLex pairs found in vocabulary:", len(pair_list))
        return float('nan')

    gold_scores = []
    pred_scores = []

    for word_i, word_j, gold_score in pair_list:
        # cosine similarity (not distance)
        sim = numpy.dot(word_vectors[word_i], word_vectors[word_j])
        gold_scores.append(gold_score)
        pred_scores.append(sim)

    spearman_rho, _ = spearmanr(gold_scores, pred_scores)
    return round(float(spearman_rho), 3)

In [10]:

# Get scores for base GloVe
print("===== Base GloVe Embeddings =====")
metrics_base, human_scores, base_scores = wordsim_analysis(embeddings_index)
print("WordSim: " + f'{metrics_base}')
print("SimLex: " + f'{simlex_analysis(embeddings_index)}')

# Get scores for counterfitted GloVe
print("\n===== Counterfitted GloVe Embeddings =====")
metrics_counter, _, counter_scores = wordsim_analysis(counter_embeddings)
print("WordSim: " + f'{metrics_counter}')
print("SimLex: " + f'{simlex_analysis(counter_embeddings)}')

# Get scores for counterfitted GloVe
print("\n===== Retrofitted GloVe Embeddings =====")
metrics_retro, _, retro_scores = wordsim_analysis(retro_embeddings)
print("WordSim: " + f'{metrics_retro}')
print("SimLex: " + f'{simlex_analysis(retro_embeddings)}')

===== Base GloVe Embeddings =====
WordSim: {'pairs_evaluated': 353, 'total_pairs': 353, 'spearman_corr': 0.4783764811477666, 'pearson_corr': 0.47879698861989267}
SimLex: 0.232

===== Counterfitted GloVe Embeddings =====
WordSim: {'pairs_evaluated': 353, 'total_pairs': 353, 'spearman_corr': 0.46999590374504935, 'pearson_corr': 0.4866601339756519}
SimLex: 0.479

===== Retrofitted GloVe Embeddings =====
WordSim: {'pairs_evaluated': 353, 'total_pairs': 353, 'spearman_corr': 0.5041477627123687, 'pearson_corr': 0.5281150687200921}
SimLex: 0.373


In [11]:
def bootstrap_spearman_diff(human_scores, base_scores, counter_scores, n_bootstrap=10000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(human_scores)
    diffs = []

    for _ in range(n_bootstrap):
        # sample indices with replacement
        idx = rng.integers(0, n, n)
        h = np.array(human_scores)[idx]
        b = np.array(base_scores)[idx]
        c = np.array(counter_scores)[idx]
        rho_base, _ = spearmanr(h, b)
        rho_counter, _ = spearmanr(h, c)
        diffs.append(rho_counter - rho_base)

    diffs = np.array(diffs)
    mean_diff = diffs.mean()
    ci_lower = np.percentile(diffs, 2.5)
    ci_upper = np.percentile(diffs, 97.5)

    # two-sided p-value for difference ≤ 0
    p_value = (np.sum(diffs <= 0) / n_bootstrap) * 2
    return {
        'mean_diff': mean_diff,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'p_value': p_value
    }

results = bootstrap_spearman_diff(human_scores, base_scores, counter_scores,
                                  n_bootstrap=10000)
print(results)

{'mean_diff': -0.008345935204921395, 'ci_lower': -0.05544354210894383, 'ci_upper': 0.03783666243814313, 'p_value': 1.2694}
